In [1]:
import pandas as pd
import numpy as np
import itertools
import torch
from torch import tensor, from_numpy, nn, optim, float32, reshape
from torch.utils.data import TensorDataset, DataLoader, TensorDataset
from torchvision import transforms
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import GridSearchCV 
from datetime import datetime


In [2]:
%pip install wandb -q
import wandb
wandb.login()

ERROR:wandb.jupyter:Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: thesanjeetc (mlcw). Use `wandb login --relogin` to force relogin


True

In [3]:
# df = pd.read_csv("housing.csv")
# df.head()

In [4]:
# df.info()

In [5]:
# df.describe()

In [6]:
# df.isnull().sum()

In [7]:
# ocean_values = df["ocean_proximity"].value_counts()
# ocean_values

In [8]:
# df.hist(bins=25,figsize=(20,10));

In [9]:
# df['total_bedrooms'].bfill(inplace=True)
# df.isnull().sum()

In [10]:
# df.columns

In [11]:
# x = df.drop(["median_house_value"], axis=1)

In [12]:
# num_cols = list(x.select_dtypes('float64').columns)
# cat_cols = list(x.select_dtypes('object').columns)

# scaler = StandardScaler()
# num_cols_trans = scaler.fit_transform(x[num_cols])
# df_num = pd.DataFrame(num_cols_trans, columns = num_cols)

# encoder = LabelEncoder()
# cat_cols_trans = encoder.fit_transform(x[cat_cols])
# df_cat = pd.DataFrame(cat_cols_trans, columns = cat_cols)

# x_transformed = pd.concat([df_num, df_cat], axis=1)
# x_transformed

In [13]:
# y = df["median_house_value"]

In [14]:
# x_train, x_test, y_train, y_test = train_test_split(x_transformed, y, test_size=0.2, random_state=42)

In [15]:
# BATCH_SIZE = 64

# train_set = TensorDataset(tensor(x_train.values), tensor(y_train.values))
# test_set = TensorDataset(tensor(x_test.values), tensor(y_test.values))

# train_dataloader = DataLoader(dataset=train_set, batch_size=BATCH_SIZE, shuffle=True)
# test_dataloader = DataLoader(dataset=test_set, batch_size=BATCH_SIZE, shuffle=True)

In [16]:
class NeuralNetwork(nn.Module):
    def __init__(self, config):
        super().__init__()

        layers = []

        for i in range(0, len(config) - 2):
          layers.append(nn.Linear(config[i], config[i + 1]))
          layers.append(nn.ReLU())

        layers.append(nn.Linear(config[-2], config[-1]))
        self.layers = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.layers(x)

In [17]:
# LEARNING_RATE=1e-4

# model = NeuralNetwork()
# loss_function = nn.MSELoss()
# optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

In [32]:
def get_datasets():
    df = pd.read_csv("housing.csv")

    x = df.drop(["median_house_value"], axis=1)
    x['total_bedrooms'].bfill(inplace=True)

    num_cols = list(x.select_dtypes('float64').columns)
    cat_cols = list(x.select_dtypes('object').columns)

    scaler = StandardScaler()
    num_cols_trans = scaler.fit_transform(x[num_cols])
    df_num = pd.DataFrame(num_cols_trans, columns = num_cols)

    encoder = LabelEncoder()
    cat_cols_trans = encoder.fit_transform(x[cat_cols])
    df_cat = pd.DataFrame(cat_cols_trans, columns = cat_cols)

    x = pd.concat([df_num, df_cat], axis=1)
    y = df["median_house_value"]

    x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)
    x_test, x_val, y_test, y_val = train_test_split(x_test, y_test, test_size=0.5, random_state=42)

    train_set = TensorDataset(tensor(x_train.values), tensor(y_train.values.ravel()))
    test_set = TensorDataset(tensor(x_test.values), tensor(y_test.values.ravel()))
    validation_set = TensorDataset(tensor(x_val.values), tensor(y_val.values.ravel()))

    print(x_train.shape, x_test.shape, x_val.shape)

    return train_set, test_set, validation_set

def get_data_loaders(batch_size):
    train_set, test_set, validation_set = get_datasets()

    train_dataloader = DataLoader(dataset=train_set, batch_size=batch_size, shuffle=True)
    test_dataloader = DataLoader(dataset=test_set, batch_size=batch_size, shuffle=True)
    validation_dataloader = DataLoader(dataset=test_set, batch_size=batch_size, shuffle=True)

    return train_dataloader, test_dataloader, validation_dataloader

def train_loop(dataloader, model, loss_fn, optimizer):
    num_batches = len(dataloader)
    size = len(dataloader.dataset)
    train_loss = 0

    for batch, (X, y) in enumerate(dataloader):
        X, y = X.float(), y.float()
        y = y.reshape((y.shape[0], 1))

        pred = model(X)
        loss = loss_fn(pred, y)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        if batch % 100 == 0:
            loss, current = loss.item(), batch * len(X)
            print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

    train_loss /= num_batches
    return train_loss


def test_loop(dataloader, model, loss_fn):
    num_batches = len(dataloader)
    test_loss = 0

    with torch.no_grad():
        for X, y in dataloader:
            X, y = X.float(), y.float()
            y = y.reshape((y.shape[0], 1))
            pred = model(X.float())
            test_loss += loss_fn(pred, y.float()).item()

    test_loss /= num_batches
    print(f"Avg loss: {test_loss:>8f} \n")
    return test_loss

def train_and_validate(num_epochs, model, loss_fn, optimiser, train_dataloader, test_dataloader, validation_dataloader):
    for i in range(num_epochs):
        print(f"Epoch {i+1}:")

        loss_train = train_loop(train_dataloader, model, loss_fn, optimiser)
        loss_valid = test_loop(validation_dataloader, model, loss_fn)

        wandb.log({
            "Training Loss": loss_train,
            "Validation Loss": loss_valid,
        })

    loss_test = test_loop(test_dataloader, model, loss_fn)

    return loss_test

In [33]:
def create_config(opts):
    learning_rate, num_epochs, batch_size, shape = opts

    model = NeuralNetwork(shape)
    loss_fn = nn.MSELoss()
    optimiser = torch.optim.Adam(model.parameters(), lr=learning_rate)

    return tuple([num_epochs, model, loss_fn, optimiser, *get_data_loaders(batch_size)])


def grid_search(params):
  options = [x for x in itertools.product(*params.values())]
  best_config = (float('inf'), None, None)

  time = datetime.now().strftime("%Y-%m-%d @ %H.%M.%S")
  project_name = "Intro2ML - {}".format(time)

  for i, opts in enumerate(options):
      config_dict = dict(zip(params.keys(), opts))
      name = "E{}".format(i)

      wandb.init(
          name = name, 
          project = project_name, 
          entity = "mlcw",
          config = config_dict,
          tags = ['Experiment']
      )

      config = create_config(opts)
      test_loss = train_and_validate(*config)

      if test_loss < best_config[0]:
          best_config = (test_loss, name, config_dict)

  wandb.finish()

  return best_config
    

In [34]:
params = {
    'learning_rate':[0.1, 0.01, 0.001],
    'num_epochs':[250, 500, 1000, 2000],
    'batch_size':[16, 32, 64],
    'shape':[[9, 32, 1], [9, 64, 1], [9, 32, 16, 1]]
}

print(grid_search(params))


/usr/local/lib/python3.7/dist-packages/sklearn/preprocessing/_label.py:115: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


(13209, 9) (1651, 9) (1652, 9)
Epoch 1:
loss: 50335621120.000000  [    0/13209]
loss: 47129903104.000000  [ 3200/13209]
loss: 42209964032.000000  [ 6400/13209]
loss: 61326598144.000000  [ 9600/13209]
loss: 67938119680.000000  [12800/13209]
Avg loss: 55344614990.769234 

Epoch 2:
loss: 39495409664.000000  [    0/13209]
loss: 91683151872.000000  [ 3200/13209]
loss: 55436394496.000000  [ 6400/13209]
loss: 60128067584.000000  [ 9600/13209]
loss: 42613616640.000000  [12800/13209]
Avg loss: 50564139598.769234 

Epoch 3:
loss: 43938476032.000000  [    0/13209]
loss: 45185986560.000000  [ 3200/13209]
loss: 57132580864.000000  [ 6400/13209]
loss: 46337384448.000000  [ 9600/13209]
loss: 69609619456.000000  [12800/13209]
Avg loss: 44207394028.307693 

Epoch 4:
loss: 39562596352.000000  [    0/13209]
loss: 43298435072.000000  [ 3200/13209]
loss: 36176674816.000000  [ 6400/13209]
loss: 52439420928.000000  [ 9600/13209]
loss: 31377776640.000000  [12800/13209]
Avg loss: 37102304610.461540 

Epoch 5:


Training Loss,█▄▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation Loss,█▄▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Training Loss,4214849845.92736
Validation Loss,4189855687.38462


/usr/local/lib/python3.7/dist-packages/sklearn/preprocessing/_label.py:115: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


(13209, 9) (1651, 9) (1652, 9)
Epoch 1:
loss: 61612294144.000000  [    0/13209]
loss: 55674322944.000000  [ 3200/13209]
loss: 57885319168.000000  [ 6400/13209]
loss: 64509763584.000000  [ 9600/13209]
loss: 51061035008.000000  [12800/13209]
Avg loss: 53807927965.538460 

Epoch 2:
loss: 47946006528.000000  [    0/13209]
loss: 71804813312.000000  [ 3200/13209]
loss: 52340543488.000000  [ 6400/13209]
loss: 43860885504.000000  [ 9600/13209]
loss: 37087600640.000000  [12800/13209]
Avg loss: 45026118813.538460 

Epoch 3:
loss: 76414476288.000000  [    0/13209]
loss: 39413145600.000000  [ 3200/13209]
loss: 43811819520.000000  [ 6400/13209]
loss: 43494711296.000000  [ 9600/13209]
loss: 40296615936.000000  [12800/13209]
Avg loss: 34574582193.230766 

Epoch 4:
loss: 29826795520.000000  [    0/13209]
loss: 24304664576.000000  [ 3200/13209]
loss: 30784397312.000000  [ 6400/13209]
loss: 23948300288.000000  [ 9600/13209]
loss: 22102495232.000000  [12800/13209]
Avg loss: 25472904113.230770 

Epoch 5:


Training Loss,█▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation Loss,█▃▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Training Loss,4155671055.18644
Validation Loss,4143144371.69231


/usr/local/lib/python3.7/dist-packages/sklearn/preprocessing/_label.py:115: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


(13209, 9) (1651, 9) (1652, 9)
Epoch 1:
loss: 59668328448.000000  [    0/13209]
loss: 54110662656.000000  [ 3200/13209]
loss: 45993779200.000000  [ 6400/13209]
loss: 27290472448.000000  [ 9600/13209]
loss: 16499459072.000000  [12800/13209]
Avg loss: 14129032881.230770 

Epoch 2:
loss: 10426599424.000000  [    0/13209]
loss: 12699386880.000000  [ 3200/13209]
loss: 13853933568.000000  [ 6400/13209]
loss: 9505921024.000000  [ 9600/13209]
loss: 11893460992.000000  [12800/13209]
Avg loss: 9114648024.615385 

Epoch 3:
loss: 9103834112.000000  [    0/13209]
loss: 5063551488.000000  [ 3200/13209]
loss: 12399975424.000000  [ 6400/13209]
loss: 5645593600.000000  [ 9600/13209]
loss: 9643177984.000000  [12800/13209]
Avg loss: 6390621846.153846 

Epoch 4:
loss: 6553416704.000000  [    0/13209]
loss: 6079997952.000000  [ 3200/13209]
loss: 4024709376.000000  [ 6400/13209]
loss: 12726452224.000000  [ 9600/13209]
loss: 5392077312.000000  [12800/13209]
Avg loss: 5114749550.769231 

Epoch 5:
loss: 390155

Training Loss,█▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Validation Loss,█▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Training Loss,3882917229.09443
Validation Loss,3869844708.92308


(3868421265.230769, 'E2', {'learning_rate': 0.01, 'num_epochs': 200, 'batch_size': 32, 'shape': [9, 32, 16, 1]})
